# Exemplar Inclusion Summary Stats (Overall)

Compute summary stats for strict exemplar filtering across **all included categories** (not only Plot A categories).

Primary report:
1. Number of valid exemplars included
2. Average category-level precision for included categories
3. Average file-level precision for included exemplars
4. Number of exemplars filtered out after strict filtering
5. Number of raters

Also reported: optional top-25-per-category capped version for comparison.


This notebook also enforces a minimum category size filter (`>= 5 exemplars/category` by default).

Per-file precision uses `actual_precision` when available (computed as `1 - error_rate`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SCRIPT_DIR = Path('.').resolve()
project_root = SCRIPT_DIR.parent.parent

# Match analysis defaults
PRECISION_THRESHOLD = 0.6
top_k_per_category = 25
min_exemplars_per_category = 5

# Inputs
INCLUDED_CATEGORIES_TXT = project_root / 'data/included_categories.txt'
PER_CLASS_VALIDATION_CSV = project_root / 'annotation/per_class_validation_data.csv'
PER_FILE_PRECISION_CSV = project_root / 'annotation/per_file_precision_data.csv'
SAMPLED_EXEMPLARS_CSV = project_root / 'annotation/sampled_object_crops_100_bucket_assignments_100ex_8subj_per_video_cap_babyview_only.csv'

for p in [INCLUDED_CATEGORIES_TXT, PER_CLASS_VALIDATION_CSV, PER_FILE_PRECISION_CSV, SAMPLED_EXEMPLARS_CSV]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print('Using threshold:', PRECISION_THRESHOLD)
print('Top-k per category (comparison only):', top_k_per_category)
print('Minimum exemplars per category:', min_exemplars_per_category)

In [ ]:
def parse_confidence_from_stem(stem):
    parts = str(stem).split('_')
    if len(parts) < 2:
        return np.nan
    try:
        return float(parts[1])
    except ValueError:
        return np.nan


def load_allowed_categories(included_txt, per_class_csv, threshold=0.6):
    included = set(line.strip().lower() for line in included_txt.read_text().splitlines() if line.strip())
    per_class = pd.read_csv(per_class_csv, usecols=['class', 'precision']).dropna(subset=['class', 'precision']).copy()
    per_class['class'] = per_class['class'].astype(str).str.strip().str.lower()
    per_class['precision'] = pd.to_numeric(per_class['precision'], errors='coerce')
    precision_allowed = set(per_class.loc[per_class['precision'] > threshold, 'class'])
    return included & precision_allowed


def load_per_file_precision(per_file_csv):
    df = pd.read_csv(per_file_csv).dropna(subset=['filename', 'class']).copy()
    if 'actual_precision' in df.columns:
        df['per_file_precision_used'] = pd.to_numeric(df['actual_precision'], errors='coerce')
    elif 'precision' in df.columns:
        df['per_file_precision_used'] = pd.to_numeric(df['precision'], errors='coerce')
    else:
        raise ValueError('Expected either `actual_precision` or `precision` column in per-file CSV.')
    df['class'] = df['class'].astype(str).str.strip().str.lower()
    df['raters'] = pd.to_numeric(df['raters'], errors='coerce')
    df['stem'] = df['filename'].map(lambda s: Path(str(s)).stem)
    return df


def load_sampled_exemplars(sampled_csv):
    df = pd.read_csv(sampled_csv).dropna(subset=['category', 'stem']).copy()
    df['category_norm'] = df['category'].astype(str).str.strip().str.lower()
    return df


def topk_per_category(df, k):
    chunks = []
    for cat, g in df.groupby('category_norm', sort=False):
        chunks.append(g.sort_values('confidence', ascending=False).head(k))
    if not chunks:
        return df.iloc[0:0].copy()
    return pd.concat(chunks, ignore_index=True)


def build_stats_rows(df, selected_pool_n, per_class_df, label):
    included_categories = sorted(set(df['category_norm'].tolist()))
    per_class_for_included = per_class_df[per_class_df['class'].isin(included_categories)].copy()

    row = {
        'scope': label,
        'n_valid_exemplars_included': int(len(df)),
        'avg_category_precision_for_included_categories': float(per_class_for_included['precision'].mean()) if len(per_class_for_included) else np.nan,
        'avg_file_level_precision_for_included_exemplars': float(df['per_file_precision_used'].mean()) if len(df) else np.nan,
        'n_filtered_out_after_strict_filtering': int(selected_pool_n - len(df)),
        'file_level_raters_unique': ', '.join(map(str, sorted({int(x) for x in df['raters'].dropna().unique()}))),
        'category_pair_raters_unique': ', '.join(map(str, sorted({int(x) for x in per_class_for_included['n_pair_raters'].dropna().unique()}))),
        'n_categories_represented': int(len(included_categories)),
    }
    return row

In [ ]:
allowed_categories = load_allowed_categories(
    INCLUDED_CATEGORIES_TXT,
    PER_CLASS_VALIDATION_CSV,
    threshold=PRECISION_THRESHOLD,
)

per_file_df = load_per_file_precision(PER_FILE_PRECISION_CSV)
sampled_df = load_sampled_exemplars(SAMPLED_EXEMPLARS_CSV)

# Apply category eligibility
sampled_selected = sampled_df[sampled_df['category_norm'].isin(allowed_categories)].copy()
per_file_selected = per_file_df[per_file_df['class'].isin(allowed_categories)].copy()

# Attach file-level precision + raters to sampled stems
merged = sampled_selected.merge(
    per_file_selected[['class', 'stem', 'per_file_precision_used', 'raters']],
    left_on=['category_norm', 'stem'],
    right_on=['class', 'stem'],
    how='left',
)

# Strict filter: file precision > threshold + parseable confidence
strict_all_valid = merged[merged['per_file_precision_used'] > PRECISION_THRESHOLD].copy()
strict_all_valid['confidence'] = strict_all_valid['stem'].map(parse_confidence_from_stem)
strict_all_valid = strict_all_valid.dropna(subset=['confidence']).copy()

# Optional comparison: cap at top-k confidence per category
# Enforce minimum exemplars per category for analysis inclusion
category_counts = strict_all_valid.groupby('category_norm').size().rename('n_exemplars').reset_index()
eligible_categories = set(category_counts.loc[category_counts['n_exemplars'] >= min_exemplars_per_category, 'category_norm'])
strict_all_valid = strict_all_valid[strict_all_valid['category_norm'].isin(eligible_categories)].copy()

strict_topk = topk_per_category(strict_all_valid, top_k_per_category)

per_class_df = pd.read_csv(PER_CLASS_VALIDATION_CSV, usecols=['class', 'precision', 'n_pair_raters']).dropna(subset=['class']).copy()
per_class_df['class'] = per_class_df['class'].astype(str).str.strip().str.lower()
per_class_df['precision'] = pd.to_numeric(per_class_df['precision'], errors='coerce')
per_class_df['n_pair_raters'] = pd.to_numeric(per_class_df['n_pair_raters'], errors='coerce')

print(f'Allowed categories after class-level threshold: {len(allowed_categories)}')
print(f'Sampled pool size after category filtering: {len(sampled_selected):,}')
print(f'Categories with >= {min_exemplars_per_category} exemplars: {len(eligible_categories)}')
print(f'Strict all-valid exemplars (overall): {len(strict_all_valid):,}')
print(f'Strict top-{top_k_per_category}/category exemplars: {len(strict_topk):,}')

In [ ]:
overall_row = build_stats_rows(
    strict_all_valid,
    selected_pool_n=len(sampled_selected),
    per_class_df=per_class_df,
    label='overall_all_valid',
)

comparison_row = build_stats_rows(
    strict_topk,
    selected_pool_n=len(sampled_selected),
    per_class_df=per_class_df,
    label=f'top_{top_k_per_category}_per_category',
)

report_df = pd.DataFrame([overall_row, comparison_row])
report_df

In [ ]:
# Requested metrics (overall only)
overall_metrics = report_df[report_df['scope'] == 'overall_all_valid'].reset_index(drop=True).loc[0]
pd.DataFrame([
    {'metric': '(1) valid exemplars included', 'value': int(overall_metrics['n_valid_exemplars_included'])},
    {'metric': '(2) avg category precision (included categories)', 'value': round(float(overall_metrics['avg_category_precision_for_included_categories']), 4)},
    {'metric': '(3) avg file-level precision (included exemplars)', 'value': round(float(overall_metrics['avg_file_level_precision_for_included_exemplars']), 4)},
    {'metric': '(4) filtered out after strict filtering', 'value': int(overall_metrics['n_filtered_out_after_strict_filtering'])},
    {'metric': '(5) file-level raters (unique)', 'value': overall_metrics['file_level_raters_unique']},
])

In [ ]:
# Optional sanity check by category (overall set)
per_category = (
    strict_all_valid.groupby('category_norm', as_index=False)
    .agg(
        n_included=('stem', 'count'),
        mean_file_precision=('precision', 'mean'),
        mean_file_raters=('raters', 'mean'),
    )
    .sort_values(['n_included', 'mean_file_precision'], ascending=[False, False])
    .reset_index(drop=True)
)
per_category.head(25)